
## Инициализация базы данных BookRecommender
# 
### Этот notebook выполняет первоначальную настройку базы данных:
### 1. Создает структуру таблиц
### 2. Импортирует пользователей из ratings.parquet
### 3. Экспортирует учетные данные в Excel
### 4. Создает резервную копию

## 1. Импорт библиотек и настройка


In [1]:

import sys
import os
import sqlite3
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Добавляем путь к проекту
project_root = os.path.dirname(os.path.abspath('.'))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Рабочая директория: {os.getcwd()}")
print(f"Корень проекта: {project_root}")


Рабочая директория: C:\Users\agent\Python\MasterCourse\DLPRJ\RecommenderSystems\notebooks
Корень проекта: C:\Users\agent\Python\MasterCourse\DLPRJ\RecommenderSystems


## 2. Проверка наличия необходимых файлов

In [4]:
# Проверяем наличие файла с оценками
ratings_path = os.path.join(project_root, 'data', 'ratings.parquet')
if os.path.exists(ratings_path):
    print(f"✅ Файл оценок найден: {ratings_path}")
    
    # Показываем информацию о файле
    try:
        ratings_df = pd.read_parquet(ratings_path)
        print(f"   📊 Количество записей: {len(ratings_df):,}")
        print(f"   👥 Уникальных пользователей: {ratings_df['user_id'].nunique():,}")
        print(f"   📚 Уникальных книг: {ratings_df['book_id'].nunique():,}")
        print(f"   📅 Диапазон оценок: {ratings_df['rating'].min()} - {ratings_df['rating'].max()}")
        
        # Показываем первые строки
        print("\nПервые 5 строк файла оценок:")
        print(ratings_df.head())
        
    except Exception as e:
        print(f"❌ Ошибка чтения файла: {e}")
else:
    print(f"⚠️  Файл оценок НЕ найден: {ratings_path}")
    print("   Пожалуйста, поместите ratings.parquet в папку data/")

✅ Файл оценок найден: C:\Users\agent\Python\MasterCourse\DLPRJ\RecommenderSystems\data\ratings.parquet
   📊 Количество записей: 5,976,479
   👥 Уникальных пользователей: 53,424
   📚 Уникальных книг: 10,000
   📅 Диапазон оценок: 1 - 5

Первые 5 строк файла оценок:
   user_id  book_id  rating
0        1      258       5
1        2     4081       4
2        2      260       5
3        2     9296       5
4        2     2318       3


In [5]:
# Проверяем наличие файла с книгами
books_path = os.path.join(project_root, 'data', 'books_result.parquet')
if os.path.exists(books_path):
    print(f"✅ Файл книг найден: {books_path}")
    
    try:
        books_df = pd.read_parquet(books_path)
        print(f"   📚 Количество книг: {len(books_df):,}")
        
        # Показываем структуру
        print(f"\nСтруктура данных книг:")
        print(f"   Колонки: {list(books_df.columns)}")
        print(f"\nПервые 3 книги:")
        print(books_df[['book_id', 'original_title', 'authors']].head(3).to_string(index=False))
        
    except Exception as e:
        print(f"❌ Ошибка чтения файла книг: {e}")
else:
    print(f"⚠️  Файл книг НЕ найден: {books_path}")


✅ Файл книг найден: C:\Users\agent\Python\MasterCourse\DLPRJ\RecommenderSystems\data\books_result.parquet
   📚 Количество книг: 10,000

Структура данных книг:
   Колонки: ['book_id', 'goodreads_book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url', 'cover_found', 'cover_source', 'cover_url', 'cover_path']

Первые 3 книги:
 book_id                           original_title                     authors
       1                         The Hunger Games             Suzanne Collins
       2 Harry Potter and the Philosopher's Stone J.K. Rowling, Mary GrandPré
       3                                 Twilight             Stephenie Meyer


In [8]:
# %% [markdown]
# ## 3. Создание базы данных

# %%
def create_database_structure(db_path='books_recommender.db'):
    """Создание структуры базы данных"""
    
    print(f"\n🛠️  Создание структуры базы данных: {db_path}")
    
    # Создаем резервную копию если файл уже существует
    if os.path.exists(db_path):
        backup_name = f"backups/users_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
        import shutil
        shutil.copy2(db_path, backup_name)
        print(f"   📦 Создана резервная копия: {backup_name}")
    
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Таблица пользователей
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                user_id INTEGER PRIMARY KEY,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        print("   ✅ Таблица 'users' создана")
        
        # Таблица оценок пользователей
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS user_ratings (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER NOT NULL,
                book_id TEXT NOT NULL,
                rating INTEGER NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (user_id) REFERENCES users (user_id)
            )
        ''')
        print("   ✅ Таблица 'user_ratings' создана")
        
        # Индексы для производительности
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_user_ratings_user_id ON user_ratings(user_id)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_user_ratings_book_id ON user_ratings(book_id)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_users_username ON users(username)')
        print("   ✅ Индексы созданы")
        
        conn.commit()
        conn.close()
        
        print(f"🎉 Структура базы данных успешно создана!")
        return True
        
    except Exception as e:
        print(f"❌ Ошибка создания структуры БД: {e}")
        return False

# Создаем структуру БД
create_database_structure()


🛠️  Создание структуры базы данных: books_recommender.db
   📦 Создана резервная копия: backups/users_backup_20251213_205815.db
   ✅ Таблица 'users' создана
   ✅ Таблица 'user_ratings' создана
   ✅ Индексы созданы
🎉 Структура базы данных успешно создана!


True

In [6]:
from tqdm import tqdm

In [7]:
# %% [markdown]
# ## 4. Импорт пользователей из ratings.parquet

# %%
def import_users_from_ratings(db_path='books_recommender.db', ratings_path='data/ratings.parquet'):
    """Импорт пользователей и их оценок из ratings.parquet"""
    
    if not os.path.exists(ratings_path):
        print(f"❌ Файл {ratings_path} не найден")
        return False
    
    print(f"\n📥 Импорт данных из {ratings_path}")
    
    try:
        # Читаем данные
        ratings_df = pd.read_parquet(ratings_path)
        
        # Получаем уникальных пользователей
        unique_users = ratings_df['user_id'].unique()
        total_users = len(unique_users)
        print(f"   👥 Найдено {total_users:,} уникальных пользователей")
        
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Очищаем таблицы (если нужно)
        cursor.execute('DELETE FROM user_ratings')
        cursor.execute('DELETE FROM users')
        cursor.execute('DELETE FROM sqlite_sequence WHERE name IN ("users", "user_ratings")')
        print("   🗑️  Старые данные удалены")
        
        # Импорт пользователей
        from werkzeug.security import generate_password_hash
        
        print("\n👤 Импорт пользователей...")
        
        users_inserted = 0
        user_data = []
        start_time = datetime.now()
        
        # Используем tqdm для отображения прогресса
        user_iterator = tqdm(unique_users, desc="Пользователи", unit="user", ncols=100)
       
        for idx, user_id in enumerate(user_iterator):
            username = f'user_{user_id}'
            email = f'user_{user_id}@bookrecommender.com'
            password_hash = generate_password_hash('12345q')  # Единый пароль
            
            user_data.append((int(user_id), username, email, password_hash))
            users_inserted += 1
            
            # Логирование каждые 50 пользователей
            if (idx + 1) % 1000 == 0:
                elapsed_time = (datetime.now() - start_time).total_seconds()
                rate = (idx + 1) / elapsed_time if elapsed_time > 0 else 0
                print(f"   📝 Обработано {idx + 1:,}/{total_users:,} пользователей "
                      f"({(idx + 1)/total_users*100:.1f}%), "
                      f"скорость: {rate:.1f} users/sec")
        
        # Массовая вставка пользователей
        print(f"\n💾 Сохранение {users_inserted:,} пользователей в БД...")
        cursor.executemany(
            'INSERT INTO users (user_id, username, email, password) VALUES (?, ?, ?, ?)',
            user_data
        )
        
        elapsed_time = (datetime.now() - start_time).total_seconds()
        print(f"   ✅ {users_inserted:,} пользователей добавлено за {elapsed_time:.1f} секунд "
              f"({users_inserted/elapsed_time:.1f} users/sec)")
        
        # Импорт оценок
        print("\n⭐ Импорт оценок пользователей...")
        ratings_inserted = 0
        batch_size = 10000
        ratings_data = []
        total_ratings = len(ratings_df)
        
        start_time_ratings = datetime.now()
        batch_counter = 0
        
        # Используем tqdm для отображения прогресса по оценкам
        
        ratings_iterator = tqdm(ratings_df.iterrows(), total=total_ratings, 
                               desc="Оценки", unit="rating", ncols=100)
       
        for idx, row in ratings_iterator:
            if isinstance(idx, tuple):  # Если используем enumerate
                row_idx, row_data = idx, row
            else:
                row_idx, row_data = idx, row
            
            ratings_data.append((
                int(row_data['user_id']),
                str(row_data['book_id']),
                int(row_data['rating'])
            ))
            ratings_inserted += 1
            
            # Пакетная вставка
            if len(ratings_data) >= batch_size:
                cursor.executemany(
                    'INSERT INTO user_ratings (user_id, book_id, rating) VALUES (?, ?, ?)',
                    ratings_data
                )
                ratings_data = []
                batch_counter += 1
                
                # Логирование каждые 50к оценок
                if batch_counter % 5 == 0:  # каждые 5 батчей = 50к оценок
                    elapsed_time = (datetime.now() - start_time_ratings).total_seconds()
                    rate = ratings_inserted / elapsed_time if elapsed_time > 0 else 0
                    print(f"   📦 Добавлено {ratings_inserted:,}/{total_ratings:,} оценок "
                          f"({ratings_inserted/total_ratings*100:.1f}%), "
                          f"скорость: {rate:.1f} ratings/sec")
        
        # Оставшиеся данные
        if ratings_data:
            cursor.executemany(
                'INSERT INTO user_ratings (user_id, book_id, rating) VALUES (?, ?, ?)',
                ratings_data
            )
        
        conn.commit()
        
        # Получаем статистику
        cursor.execute('SELECT COUNT(*) FROM users')
        db_total_users = cursor.fetchone()[0]
        
        cursor.execute('SELECT COUNT(*) FROM user_ratings')
        db_total_ratings = cursor.fetchone()[0]
        
        conn.close()
        
        total_elapsed_time = (datetime.now() - start_time).total_seconds()
        
        print(f"\n{'='*60}")
        print("📊 ИМПОРТ ЗАВЕРШЕН:")
        print('='*60)
        print(f"   👤 Пользователей в БД: {db_total_users:,}")
        print(f"   ⭐ Оценок в БД: {db_total_ratings:,}")
        print(f"   ⏱️  Общее время: {total_elapsed_time:.1f} секунд")
        print(f"   🚀 Скорость импорта: {total_ratings/total_elapsed_time:.1f} ratings/sec")
        print('='*60)
        
        return True
        
    except Exception as e:
        print(f"❌ Ошибка импорта: {e}")
        import traceback
        traceback.print_exc()
        return False

# Запускаем импорт
import_users_from_ratings(ratings_path=ratings_path)


📥 Импорт данных из C:\Users\agent\Python\MasterCourse\DLPRJ\RecommenderSystems\data\ratings.parquet
   👥 Найдено 53,424 уникальных пользователей
   🗑️  Старые данные удалены

👤 Импорт пользователей...


Пользователи:   2%|▊                                       | 1002/53424 [01:12<1:02:58, 13.87user/s]

   📝 Обработано 1,000/53,424 пользователей (1.9%), скорость: 13.9 users/sec


Пользователи:   4%|█▍                                      | 2002/53424 [02:25<1:08:30, 12.51user/s]

   📝 Обработано 2,000/53,424 пользователей (3.7%), скорость: 13.8 users/sec


...

Оценки: 100%|█████████████████████████████████████▊| 5955021/5976479 [03:39<00:00, 23861.21rating/s]

   📦 Добавлено 5,950,000/5,976,479 оценок (99.6%), скорость: 27071.7 ratings/sec


Оценки: 100%|██████████████████████████████████████| 5976479/5976479 [03:40<00:00, 27061.97rating/s]



📊 ИМПОРТ ЗАВЕРШЕН:
   👤 Пользователей в БД: 53,424
   ⭐ Оценок в БД: 5,976,479
   ⏱️  Общее время: 4068.3 секунд
   🚀 Скорость импорта: 1469.0 ratings/sec


True

In [8]:
# %% [markdown]
# ## 5. Экспорт учетных данных в Excel

# %%
def export_users_to_excel(db_path='books_recommender.db', output_path='users_credentials.xlsx'):
    """Экспорт данных пользователей в Excel"""
    
    print(f"\n📊 Экспорт данных пользователей в Excel...")
    
    try:
        conn = sqlite3.connect(db_path)
        
        # Получаем данные пользователей
        query = '''
            SELECT 
                user_id,
                username,
                email,
                '12345q' as password,
                created_at
            FROM users
            ORDER BY user_id
        '''
        
        users_df = pd.read_sql_query(query, conn)
        conn.close()
        
        # Экспорт в Excel
        users_df.to_excel(output_path, index=False)
        
        print(f"✅ Данные экспортированы в: {output_path}")
        print(f"   📄 Всего записей: {len(users_df):,}")
        
        # Создаем дополнительный CSV для удобства
        csv_path = output_path.replace('.xlsx', '.csv')
        users_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"   📄 Также создан CSV файл: {csv_path}")
        
        # Показываем первые 10 записей
        print(f"\nПервые 10 пользователей:")
        print(users_df.head(10).to_string(index=False))
        
        # Статистика
        print(f"\n📈 Статистика:")
        print(f"   🏷️  Имя пользователя: user_1 - user_{users_df['user_id'].max()}")
        print(f"   🔑 Пароль для всех: 12345q")
        print(f"   📧 Email: user_X@bookrecommender.com")
        
        return True
        
    except Exception as e:
        print(f"❌ Ошибка экспорта: {e}")
        return False

# Экспортируем данные
export_users_to_excel()


📊 Экспорт данных пользователей в Excel...
✅ Данные экспортированы в: users_credentials.xlsx
   📄 Всего записей: 53,424
   📄 Также создан CSV файл: users_credentials.csv

Первые 10 пользователей:
 user_id username                       email password          created_at
       1   user_1  user_1@bookrecommender.com   12345q 2025-12-13 19:03:20
       2   user_2  user_2@bookrecommender.com   12345q 2025-12-13 19:03:20
       3   user_3  user_3@bookrecommender.com   12345q 2025-12-13 19:03:20
       4   user_4  user_4@bookrecommender.com   12345q 2025-12-13 19:03:20
       5   user_5  user_5@bookrecommender.com   12345q 2025-12-13 19:03:20
       6   user_6  user_6@bookrecommender.com   12345q 2025-12-13 19:03:20
       7   user_7  user_7@bookrecommender.com   12345q 2025-12-13 19:03:20
       8   user_8  user_8@bookrecommender.com   12345q 2025-12-13 19:03:20
       9   user_9  user_9@bookrecommender.com   12345q 2025-12-13 19:03:20
      10  user_10 user_10@bookrecommender.com   12345q

True

In [9]:
# %% [markdown]
# ## 6. Проверка целостности данных

# %%
def verify_database_integrity(db_path='books_recommender.db'):
    """Проверка целостности базы данных"""
    
    print(f"\n🔍 Проверка целостности базы данных...")
    
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Проверяем таблицы
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = cursor.fetchall()
        print(f"   ✅ Найдено таблиц: {len(tables)}")
        for table in tables:
            print(f"      - {table[0]}")
        
        # Статистика по таблицам
        print(f"\n📊 Статистика по таблицам:")
        
        for table_name in ['users', 'user_ratings']:
            cursor.execute(f'SELECT COUNT(*) FROM {table_name}')
            count = cursor.fetchone()[0]
            print(f"   📋 {table_name}: {count:,} записей")
        
        # Проверяем связи (foreign keys)
        cursor.execute('PRAGMA foreign_key_check')
        fk_issues = cursor.fetchall()
        if not fk_issues:
            print("   ✅ Проверка внешних ключей: OK")
        else:
            print("   ⚠️  Проблемы с внешними ключами:")
            for issue in fk_issues:
                print(f"      {issue}")
        
        # Пример запроса для проверки данных
        print(f"\n👤 Пример данных пользователя (user_id=1):")
        cursor.execute('''
            SELECT u.user_id, u.username, COUNT(ur.book_id) as books_rated, 
                   AVG(ur.rating) as avg_rating
            FROM users u
            LEFT JOIN user_ratings ur ON u.user_id = ur.user_id
            WHERE u.user_id = 1
            GROUP BY u.user_id
        ''')
        user_example = cursor.fetchone()
        if user_example:
            print(f"   User ID: {user_example[0]}")
            print(f"   Username: {user_example[1]}")
            print(f"   Книг оценено: {user_example[2]}")
            print(f"   Средняя оценка: {user_example[3]:.2f}")
        
        conn.close()
        
        print(f"\n🎉 Проверка целостности завершена успешно!")
        return True
        
    except Exception as e:
        print(f"❌ Ошибка проверки: {e}")
        return False

# Проверяем базу данных
verify_database_integrity()


🔍 Проверка целостности базы данных...
   ✅ Найдено таблиц: 3
      - users
      - user_ratings
      - sqlite_sequence

📊 Статистика по таблицам:
   📋 users: 53,424 записей
   📋 user_ratings: 5,976,479 записей
   ✅ Проверка внешних ключей: OK

👤 Пример данных пользователя (user_id=1):
   User ID: 1
   Username: user_1
   Книг оценено: 117
   Средняя оценка: 3.59

🎉 Проверка целостности завершена успешно!


True

In [ ]:

# %% [markdown]
# ## 7. Создание файла конфигурации для приложения

# %%
def create_config_file():
    """Создание конфигурационного файла для приложения"""
    
    config_content = '''# Конфигурация BookRecommender
# Автоматически сгенерировано: {date}

# Настройки базы данных
DATABASE_PATH = 'books_recommender.db'

# Настройки модели
MODEL_PATH = 'models'

# Настройки приложения
SECRET_KEY = '854548'
DEBUG = True

# Количество рекомендаций
DEFAULT_RECOMMENDATIONS_COUNT = 10
MAX_RECOMMENDATIONS_COUNT = 50

# Пути к данным
BOOKS_DATA_PATH = 'data/books_with_more_covers.csv'
RATINGS_DATA_PATH = 'data/ratings.parquet'
'''.format(date=datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
    
    with open('app_config.py', 'w', encoding='utf-8') as f:
        f.write(config_content)
    
    print(f"✅ Конфигурационный файл создан: app_config.py")
    return True

#create_config_file()
